# NFL: ratings, spreads and season forecasts, as of any moment

Isodrome's `nflverse.games` connector publishes the raw NFL data in `isodrome/public`, readable
by anyone signed in:

- `nfl_schedule`: every version of every game's row since 1999;
- `nfl_results`: final scores;
- `nfl_team_games`: each team's stats per game;
- `nfl_injuries`: injury reports.

Each feed has the kickoff as event time, and as publication time when each fact became known.

The models are not public: they are the example project `isodrome-server/docs/examples/nfl`,
seven **stateful transformers** you deploy into **your own namespace**. Each learns from games
already played and writes a feed there: `nfl_elo` (ratings), `nfl_spread` (a predicted margin
for every unplayed game) and `nfl_forecast` (5,000 season simulations after every game), among
others.

**Setup:** `pip install 'isodrome[notebook]'`, then `iso login` once in a terminal.

In [ ]:
import os
import isodrome as iso
import pandas as pd

client = iso.connect()
schedule = client.feed("isodrome/public/nfl_schedule")
SOURCE = schedule.attribution        # nflverse, CC BY 4.0: credit it wherever the data is shown

def credited(df):
    """Shows a table of NFL data with its source's credit."""
    display(df)
    print(f"Data: {SOURCE}")

schedule

## 1. Deploy the models into your namespace

Upstream first, so each one backfills from its upstream's complete history. From a terminal, in
an `isodrome-server` checkout:

```bash
iso transformer deploy docs/examples/nfl --name nfl-elo
iso transformer deploy docs/examples/nfl --name nfl-pass-yards
iso transformer deploy docs/examples/nfl --name nfl-rush-yards
iso transformer deploy docs/examples/nfl --name nfl-spread
iso transformer deploy docs/examples/nfl --name nfl-forecast
```

Each `deploy` waits for:

- the build: installing numpy, then verifying on recent games and the project's fixtures. For a
  stateful model, verification replays its history, which takes minutes;
- then the backfill: every game since 1999, replayed in publication order.

Only then is it live. The next cell does the same from Python, deploying only what is missing.
Set `NFL_EXAMPLE` to the project's directory.

In [ ]:
PROJECT = os.environ.get("NFL_EXAMPLE", "../../isodrome-server/docs/examples/nfl")
MODELS = ["nfl-elo", "nfl-pass-yards", "nfl-rush-yards", "nfl-spread", "nfl-forecast"]
deployed = set(client.transformers().get("transformer", pd.Series(dtype=str)).str.split("/").str[-1])
for name in MODELS:
    if name not in deployed:
        client.deploy_transformer(PROJECT, name=name)       # waits until it is live
client.transformers()

In [ ]:
client.transformer("nfl-forecast")

## 2. Ratings now

The newest row per team (`latest`) of `nfl_elo`, in your namespace.

In [ ]:
ratings = iso.latest("nfl_elo", columns=["team", "elo", "season", "rated_at"]).to_df()
credited(ratings.sort_values("elo", ascending=False).head(10).reset_index(drop=True))

## 3. This week's games: the model's spread against the market's

`nfl_spread` writes a version of every unplayed game's prediction whenever something it depends
on changes. For the games kicking off in the next week, keep the most recently published version
of each (`dedupe` by key, by publication order).

In [ ]:
now = pd.Timestamp.now(tz="UTC")
columns = ["kickoff", "away_team", "home_team", "spread", "market_spread", "home_win_probability"]
week = iso.read("nfl_spread", start=now, end=now + pd.Timedelta("7d"), dedupe=["_key"],
                dedupe_order="publication", columns=columns).to_df()
credited(week.sort_values("kickoff").reset_index(drop=True))

## 4. One team's season, week by week

`nfl_forecast` has a row per team after every game: `week` is the week just played (0: the
preseason forecast), `week_complete` marks the last update of that week.

In [ ]:
TEAM = "KC"
season = int(ratings["season"].max())
odds = iso.read("nfl_forecast", where={"team": TEAM, "season": season, "week_complete": True},
                columns=["week", "wins", "losses", "playoffs", "division_title", "champion"]).to_df()
odds = odds.sort_values("week").set_index("week")
odds[["playoffs", "division_title", "champion"]].plot(
    title=f"{TEAM} {season}: chances after each week", figsize=(9, 3), ylim=(0, 1))
print(f"Data: {SOURCE}")

## 5. The league as it stood on a past date

Every forecast row is kept with when it was published, so "how did things look on 30 November
2025?" is one query: `as_of` that day, on the **published** basis, one row per team.

Two kinds of "as of" are available. The **published** basis gives what the current model computed
with what was known then (a redeploy recomputes history). The **received** basis gives what was
actually published at the time.

In [ ]:
then = pd.Timestamp("2025-11-30", tz="UTC")
league = iso.read("nfl_forecast", where={"season": 2025}, as_of=then, as_of_basis="published",
                  dedupe=["_key"], dedupe_order="publication",
                  columns=["team", "wins", "losses", "playoffs", "champion"]).to_df()
credited(league.sort_values("champion", ascending=False).head(10).reset_index(drop=True))

## 6. A backtest computes exactly what live did

`run_transformer` replays `nfl-forecast`'s triggers over a range into a feed of its own, its
state warmed up from the very first game. Its forecasts match what the live transformer wrote,
row for row.

In [ ]:
run = client.run_transformer("nfl-forecast", "2025-09-01", "2026-03-01", into="forecast_2025")
ref = f"{run.output.namespace}/{run.output.name}@{run.output.version}"
cols = ["as_of", "week", "playoffs", "champion"]
backtest = iso.read(ref, where={"team": TEAM}, columns=cols).to_df()
live = iso.read("nfl_forecast", where={"team": TEAM, "season": 2025}, columns=cols).to_df()
joined = backtest.merge(live, on=["as_of", "week"], suffixes=("_backtest", "_live"))
same = (joined["champion_backtest"] == joined["champion_live"]).all()
print(f"{len(joined)} forecasts compared; identical: {same}")
credited(joined.tail(5))